In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# Load and clean
df = pd.read_csv('RFM_transactions.csv', parse_dates=['InvoiceDate'])

In [3]:
df.head(3)

,InvoiceNo,CustomerID,InvoiceDate,Quantity,UnitPrice
0,10001,C042,2024-04-27,1,247.04
1,10002,NaN,2024-03-03,3,943.88
2,10003,C082,2024-01-17,-1,656.65


In [4]:
df = df.dropna(subset=['CustomerID'])

In [5]:
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

In [6]:
df.info()
df.shape

<class 'pandas.DataFrame'>
Index: 331 entries, 0 to 497
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    331 non-null    int64         
 1   CustomerID   331 non-null    str           
 2   InvoiceDate  331 non-null    datetime64[us]
 3   Quantity     331 non-null    int64         
 4   UnitPrice    331 non-null    float64       
dtypes: datetime64[us](1), float64(1), int64(2), str(1)
memory usage: 15.5 KB


(331, 5)

In [7]:
#Feature Engineer: Total Spend
df['Total Spend'] = df['Quantity'] * df['UnitPrice']
df

,InvoiceNo,CustomerID,InvoiceDate,Quantity,UnitPrice,Total Spend
0,10001,C042,2024-04-27,1,247.04,247.04
3,10004,C067,2024-03-13,7,615.58,4309.06
4,10005,C018,2024-02-02,2,522.43,1044.86
5,10006,C018,2024-04-26,8,246.06,1968.48
6,10007,C007,2024-03-24,7,193.00,1351.00
...,...,...,...,...,...,...
492,10493,C031,2024-01-29,5,403.09,2015.45
493,10494,C021,2024-02-07,5,602.59,3012.95
494,10495,C052,2024-02-26,8,506.83,4054.64
496,10497,C065,2024-04-10,6,153.71,922.26


In [8]:
#Statistical Outlier Removal
Q1 = df['Total Spend'].quantile(0.25)
Q3 = df['Total Spend'].quantile(0.75)
IQR = Q3 - Q1
df = df[((df['Total Spend'] < (Q1 - 1.5*IQR)) | (df['Total Spend'] > (Q3 + 1.5*IQR)))]

In [9]:
df

,InvoiceNo,CustomerID,InvoiceDate,Quantity,UnitPrice,Total Spend
359,10360,C064,2024-01-19,9,945.72,8511.48
375,10376,C036,2024-04-16,9,999.43,8994.87
423,10424,C055,2024-01-22,9,956.41,8607.69


In [10]:
#Aggregating RFM Metrics
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

In [11]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [12]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'count',
    'Total Spend': 'sum'
})

In [13]:
rfm.columns = ['Recency', 'Frequency', 'Monetary']

In [14]:
print(rfm.dtypes)

Recency        int64
Frequency      int64
Monetary     float64
dtype: object


In [15]:
rfm = rfm.dropna()
rfm = rfm[rfm['Recency'] >= 0]

In [18]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1], duplicates='drop')
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4])

In [19]:
# Simple segment categorization mapping function
def segment_me(df):
    r = df['R_Score']
    f = df['F_Score']
    if r >= 3 and f >= 3: return 'Champions'
    elif r <= 2 and f >= 3: return 'At Risk'
    elif r >= 3 and f <= 2: return 'Loyal/New'
    else: return 'Hibernating'

rfm['Segment'] = rfm.apply(segment_me, axis=1)


In [21]:
df

,InvoiceNo,CustomerID,InvoiceDate,Quantity,UnitPrice,Total Spend
359,10360,C064,2024-01-19,9,945.72,8511.48
375,10376,C036,2024-04-16,9,999.43,8994.87
423,10424,C055,2024-01-22,9,956.41,8607.69
